# Extend FedNCF

This notebook changes FedNCF in two ways, by subclassing the classes the library already provides. The round loop, the dataset and the evaluator stay as they are.

The first change trains with a pairwise ranking loss, by subclassing the client. The server is `FedNCF`, and it still aggregates the same three groups.

The second change keeps the MLP and the prediction layer on each client. The server still holds the item embeddings and still aggregates them. The server subclass declares where those groups live. The client subclass loads the client's copy for training and for scoring.

The table is synthetic and each run is two rounds, so both finish on a CPU. `examples/quickstart.ipynb` is the 100-round MovieLens configuration of FedNCF as published. `examples/custom_aggregator.ipynb` adds an aggregator and leaves the method unchanged.

FedNCF (Perifanis and Efraimidis, *Knowledge-Based Systems*, 2022) is neural collaborative filtering with one user per client. In the published setup the server holds the item embeddings and the shared network, and each client holds that user's embeddings.

## Install

FedRecLab is installed from GitHub. The command also installs NumPy, PyTorch, pandas, tqdm and Cornac. PyTorch is a large download. Run this cell before any import. The notebook does not use the repository working directory, so the same cell works in Colab from a fresh environment.

In [ ]:
%pip install "git+https://github.com/aymenkhs/FedRecLab.git"

## A small dataset

`load_dataframe_dataset` reads a pandas DataFrame and returns an `InteractionDataset`. That object is the only copy of the interactions. It remaps user and item ids to contiguous integers starting at 0, splits, and assigns clients. The frame needs `user_id` and `item_id`. `rating` and `timestamp` are optional. A chronological split reads `timestamp`.

The frame below has 12 users, 20 items and 8 distinct items per user. Ratings are integers from 1 to 5. Timestamps increase within each user. The split is stratified by user and chronological: 10% validation, 20% test. Cornac turns each fraction into a count with `ceil`, separately for every user. `ceil(0.1 * 8) + ceil(0.2 * 8)` is 3, so five interactions per user stay in training. `CrossDevicePartitionStrategy` assigns each training user to their own client. `seed=42` fixes the split. `k_core` and the two minimums are 0, which leaves every row in the table.

Both runs below share this dataset. Each run builds its own client, selector and aggregators. Those objects are single-use.

In [ ]:
import numpy as np
import pandas as pd

from fedreclab.data.client_partitioning import CrossDevicePartitionStrategy
from fedreclab.data.load_data import load_dataframe_dataset
from fedreclab.data.split import StratifiedSplitStrategy

SEED = 42
rng = np.random.default_rng(SEED)
n_users = 12
n_items = 20
per_user = 8

user_ids = np.repeat(np.arange(n_users), per_user)
item_pool = np.arange(n_items)
item_ids = np.concatenate(
    [rng.choice(item_pool, size=per_user, replace=False) for _ in range(n_users)]
)
ratings = rng.integers(1, 6, size=user_ids.shape[0]).astype(float)
base = rng.integers(1_600_000_000, 1_700_000_000, size=n_users)
offsets = np.sort(rng.integers(0, 10_000_000, size=(n_users, per_user)), axis=1)
timestamps = (base[:, None] + offsets).ravel()

interactions = pd.DataFrame(
    {
        "user_id": user_ids,
        "item_id": item_ids,
        "rating": ratings,
        "timestamp": timestamps,
    }
)

dataset = load_dataframe_dataset(
    interactions,
    splitting_strategy=StratifiedSplitStrategy(
        include_validation=True,
        include_test=True,
        val_size=0.1,
        test_size=0.2,
        chrono=True,
        by="user",
        seed=SEED,
    ),
    client_partitioning_strategy=CrossDevicePartitionStrategy(),
    include_validation=True,
    include_test=True,
    seed=SEED,
    k_core=0,
    min_user_interactions=0,
    min_item_interactions=0,
)
print(
    f"{dataset.num_users} users, {dataset.num_items} items, "
    f"{dataset.num_interactions} interactions, {dataset.num_clients} clients"
)
print(dataset.interaction_data["split"].value_counts())

## Where a change goes

FedNCF exchanges three groups. Each group is a `ParamIO`: who holds it, whether it is exchanged, where the aggregated value is written, and whether the upload is a dense tensor or a set of rows.

- `item_embeddings` is server-resident and row layout. A client uploads the item rows it trained on, as `{rows, values}`. `SimpleAvg` averages each row over the clients that sent it.
- `shared_layers` is the MLP, a dense state dict. `FedAvg` averages it, weighted by how many training rows the client reported.
- `output_weights` is the prediction layer, also dense, also `FedAvg`.

User embeddings are already client-resident. They live in `ClientState` and are not one of these groups.

One client-algorithm object is shared by every client. The server sends the server-resident groups on `ClientTrainingRequest.global_payload`. The client returns one package per group it trained. The server aggregates a group when `exchange` names an update, such as `"params"`.

`local_train` is the method to replace when the training step changes. Ranking calls `compute_scores` once per user, on that user's candidate items, so a change in the parameters used at scoring time belongs there too. A server subclass is how a group's `ParamIO` changes, and how a value is published on `extra_payload` for every client to read.

## Pairwise loss

`FedNCFClientAlgorithm.local_train` builds one epoch as a list of rows. For each observed item it writes one positive row and then `neg_samples` negative rows, drawn from outside that user's training items. The pointwise loss scores each row on its own. With the library default that loss is binary cross-entropy, and the positive label is 1.

Bayesian Personalised Ranking uses those rows as pairs. `stack_bpr_pairs` repeats each positive once per negative that follows it, so row `t` of the two arrays is one pair: the same user, the observed item, and one unobserved item. The loss on a batch of pairs is the mean of `-log sigmoid(s_ui - s_uj)`. The rating column is left unread. Ranking still calls the parent `compute_scores`, which returns the raw scores.

`num_training_examples` counts positives and negatives, the same count the pointwise client reports, so a `FedAvg` weight stays proportional to the number of observed items. One optimiser step averages up to `batch_size` pairs.

The sampler, the user-embedding injection, the optimiser and the upload are the same calls the pointwise client makes. The inner step scores two rows and subtracts them, so this subclass overrides `local_train` and keeps that surrounding sequence.

## Align the sampled rows

`_build_epoch_samples` writes one positive and then `neg_samples` negatives, and repeats that block for every positive. `stack_bpr_pairs` turns one block into `neg_samples` aligned pairs.

In [ ]:
import numpy as np

def stack_bpr_pairs(samples_x: np.ndarray, neg_samples: int) -> tuple[np.ndarray, np.ndarray]:
    """Align interleaved training rows into BPR (positive, negative) pairs.

    ``FedNCFClientAlgorithm._build_epoch_samples`` writes one positive row
    and then ``neg_samples`` negative rows, and repeats that block for every
    positive. Each positive is repeated once per negative so row ``t`` of
    the two returned arrays is one training pair.

    Parameters
    ----------
    samples_x : ndarray
        Integer array of shape ``(n_pos * (1 + neg_samples), 2)``. Columns
        are ``[local_user_id, item_id]``. An empty array with shape
        ``(0, 2)`` returns two empty pair arrays.
    neg_samples : int
        Negatives that follow each positive. Must be at least 1.

    Returns
    -------
    pos_pairs, neg_pairs : ndarray
        Arrays of shape ``(n_pos * neg_samples, 2)`` with the same columns
        as ``samples_x``. ``pos_pairs[t]`` is the positive aligned with
        ``neg_pairs[t]``.

    Raises
    ------
    ValueError
        If ``neg_samples < 1`` or the row count is not divisible by
        ``1 + neg_samples``.

    Examples
    --------
    >>> x = np.array([[0, 1], [0, 2], [0, 3], [1, 4], [1, 5], [1, 6]])
    >>> pos, neg = stack_bpr_pairs(x, 2)
    >>> pos.tolist()
    [[0, 1], [0, 1], [1, 4], [1, 4]]
    >>> neg.tolist()
    [[0, 2], [0, 3], [1, 5], [1, 6]]
    """
    if neg_samples < 1:
        raise ValueError(f"neg_samples must be >= 1, got {neg_samples}")
    stride = 1 + neg_samples
    if len(samples_x) == 0:
        empty = np.zeros((0, 2), dtype=np.int64)
        return empty, empty
    if len(samples_x) % stride != 0:
        raise ValueError(
            f"expected rows divisible by {stride}, got {len(samples_x)}"
        )
    blocks = np.asarray(samples_x).reshape(-1, stride, 2)
    pos_pairs = np.repeat(blocks[:, 0, :], neg_samples, axis=0)
    neg_pairs = blocks[:, 1:, :].reshape(-1, 2)
    return pos_pairs, neg_pairs

## Check the alignment

Six rows and two negatives per positive. Each positive item is repeated once for each negative that followed it. An empty input stays empty.

In [ ]:
x = np.array([[0, 1], [0, 2], [0, 3], [1, 4], [1, 5], [1, 6]])
pos, neg = stack_bpr_pairs(x, 2)
print("positives:\n", pos)
print("negatives:\n", neg)
assert pos.tolist() == [[0, 1], [0, 1], [1, 4], [1, 4]]
assert neg.tolist() == [[0, 2], [0, 3], [1, 5], [1, 6]]
assert stack_bpr_pairs(np.zeros((0, 2), dtype=np.int64), 2)[0].shape == (0, 2)

## The pairwise client

Three places in `local_train` are the whole change. The call to `_build_epoch_samples` is the parent's sampler. `stack_bpr_pairs` and the two `model(...)` calls are the new step. The block that builds `updates_packages` and the new `ClientStateBase` is the parent's upload: three groups, item rows limited to the items this client touched, and the user embeddings written back onto the client.

In [ ]:
import time

import numpy as np
import torch
import torch.nn.functional as F

from fedreclab.federated.base_client import (
    ClientStateBase,
    ClientTrainingRequest,
    ClientTrainingResponse,
)
from fedreclab.federated.utils import restore_named_optimizer_state
from fedreclab.methods.fedncf.fedncf_client import FedNCFClientAlgorithm

class FedNCFBPRClientAlgorithm(FedNCFClientAlgorithm):
    """FedNCF client that optimises a pairwise BPR loss.

    Each training positive is paired with ``neg_samples`` items drawn by
    the parent sampler from outside that user's training items. The loss on
    a batch of pairs is the mean of ``-log sigmoid(s_ui - s_uj)``. Ratings
    are not read. Ranking still calls the parent ``compute_scores`` on the
    raw scores.

    ``num_training_examples`` counts positives and negatives, the same count
    the pointwise client reports, so FedAvg weights stay proportional to the
    number of training positives. A step averages up to ``batch_size`` pairs.

    Parameters
    ----------
    *args, **kwargs
        Forwarded to ``FedNCFClientAlgorithm``. ``neg_samples`` must be at
        least 1. After the parent constructor returns, ``loss`` is ``"bpr"``.

    Raises
    ------
    ValueError
        If ``neg_samples < 1``, or if the parent constructor rejects an
        argument.

    Examples
    --------
    >>> FedNCFBPRClientAlgorithm(predictive_factor=4, neg_samples=4).loss
    'bpr'
    >>> FedNCFBPRClientAlgorithm(neg_samples=0)
    Traceback (most recent call last):
    ...
    ValueError: neg_samples must be >= 1 for BPR, got 0
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        if self.neg_samples < 1:
            raise ValueError(
                f"neg_samples must be >= 1 for BPR, got {self.neg_samples}"
            )
        self.loss = "bpr"

    def local_train(self, training_request: ClientTrainingRequest) -> ClientTrainingResponse:
        """Run the FedNCF local loop with a pairwise BPR loss.

        Setup, the negative sampler, the optimiser, and the upload match
        ``FedNCFClientAlgorithm.local_train``. The inner step scores the
        positive and the negative separately and applies BPR.

        Parameters
        ----------
        training_request : ClientTrainingRequest
            Standard FedNCF request, including the server payload and the
            client's training rows.

        Returns
        -------
        ClientTrainingResponse
            Updated user embeddings, all three server groups, per-epoch
            mean losses, and a training-example count of positives plus
            negatives.

        Raises
        ------
        ValueError
            If the sampler's row layout cannot be split into pairs.
        """
        t0 = time.time()
        self.init_model_skeleton()
        model = self._skeleton
        device = self.device

        client_id = training_request.client_id
        client_state = training_request.client_state
        training_data = training_request.training_data
        user_id_to_local = client_state.local_parameters["user_id_to_local"]

        model.load_server_weights(training_request.global_payload)
        model.train()

        named_params, gmf_user, mlp_user = self._inject_user_parameters(client_state)
        optimizer = torch.optim.Adam(
            list(model.parameters()) + [gmf_user, mlp_user], lr=self.lr
        )
        restore_named_optimizer_state(
            optimizer, named_params, client_state.optimizer_state
        )

        user_col, item_col = "user_id", "item_id"
        rating_col = "rating" if "rating" in training_data.columns else None

        pos_global_uids = training_data[user_col].values
        pos_item_ids = training_data[item_col].values
        pos_ratings = (
            training_data[rating_col].values
            if rating_col
            else np.ones(len(training_data), dtype=np.float32)
        )
        pos_local_uids = np.array([user_id_to_local[u] for u in pos_global_uids])

        user_pos_items: dict[int, set] = {}
        for local_uid, iid in zip(pos_local_uids, pos_item_ids):
            user_pos_items.setdefault(int(local_uid), set()).add(int(iid))

        all_losses: list[float] = []
        num_training_examples = 0
        touched_items = set(int(i) for i in pos_item_ids)
        if training_request.seed is None:
            sample_rng = None
        else:
            sample_rng = np.random.default_rng(
                [
                    training_request.seed,
                    training_request.client_id,
                    training_request.current_round,
                ]
            )

        for _ in range(self.local_epochs):
            samples_x, _samples_y = self._build_epoch_samples(
                pos_local_uids,
                pos_item_ids,
                pos_ratings,
                user_pos_items,
                sample_rng,
            )
            # Same count as the pointwise client (positives plus negatives),
            # so FedAvg weights follow the number of training positives.
            num_training_examples += len(samples_x)
            if self.item_upload_include_negatives:
                touched_items.update(int(i) for i in samples_x[:, 1])

            pos_pairs, neg_pairs = stack_bpr_pairs(samples_x, self.neg_samples)
            n_pairs = len(pos_pairs)
            if n_pairs:
                perm = (
                    np.random.permutation(n_pairs)
                    if sample_rng is None
                    else sample_rng.permutation(n_pairs)
                )
                pos_pairs = pos_pairs[perm]
                neg_pairs = neg_pairs[perm]

            epoch_loss = 0.0
            num_batches = 0
            for start in range(0, n_pairs, self.batch_size):
                bp = torch.tensor(
                    pos_pairs[start:start + self.batch_size],
                    dtype=torch.long,
                    device=device,
                )
                bn = torch.tensor(
                    neg_pairs[start:start + self.batch_size],
                    dtype=torch.long,
                    device=device,
                )
                diff = model(bp, gmf_user, mlp_user) - model(bn, gmf_user, mlp_user)
                loss = -F.logsigmoid(diff).mean()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()
                num_batches += 1

            if num_batches > 0:
                all_losses.append(epoch_loss / num_batches)

        server_groups = model.get_server_state_groups()
        server_groups["item_embeddings"] = self._item_embedding_upload(
            training_request, server_groups["item_embeddings"], touched_items
        )
        updates_packages = {
            group_name: self._update_package(training_request, group_name, group_value)
            for group_name, group_value in server_groups.items()
        }
        updated_state = ClientStateBase(
            client_id=client_id,
            local_parameters={
                "user_embeddings": {
                    "gmf_user_embeddings.weight": gmf_user.detach().cpu().clone(),
                    "mlp_user_embeddings.weight": mlp_user.detach().cpu().clone(),
                },
                "user_id_to_local": user_id_to_local,
            },
            optimizer_state=self._scoped_optimizer_state(optimizer, named_params),
        )
        return ClientTrainingResponse(
            client_id=client_id,
            updated_client_state=updated_state,
            updates_packages=updates_packages,
            training_losses=all_losses,
            training_duration=time.time() - t0,
            num_training_examples=num_training_examples,
        )

## Check the constructor

The parent arguments are unchanged. `neg_samples` has to be at least 1, because each pair needs a negative. After construction, `loss` is `"bpr"`.

In [ ]:
print(FedNCFBPRClientAlgorithm(predictive_factor=4, neg_samples=4).loss)
try:
    FedNCFBPRClientAlgorithm(neg_samples=0)
except ValueError as exc:
    print(exc)
else:
    raise AssertionError("neg_samples=0 should fail")

## Train with the pairwise client

`predictive_factor` is the latent width and is the same value on the client and the server. One local epoch and width 4 keep the demo short. `RandomClientSelection` draws 4 of the 12 clients each round. `seed=42` on the server seeds PyTorch and, because the selector was built without its own seed, the client draws as well. The second run uses the same draw size so that some clients are left out; that matters for the layers kept on the client below.

`item_embeddings` uses `SimpleAvg`. Clients upload those rows in the clear, so the server sees which items each selected client updated. `shared_layers` and `output_weights` use `FedAvg`.

The server is built without an evaluator, so the two rounds do no validation. The losses printed afterwards are the per-client means from the last round.

In [ ]:
from fedreclab.federated.aggregation import FedAvg, SimpleAvg
from fedreclab.federated.client_selection import RandomClientSelection
from fedreclab.methods.fedncf.fedncf import FedNCF

bpr_client = FedNCFBPRClientAlgorithm(
    predictive_factor=4,
    local_epochs=1,
    lr=5e-4,
    batch_size=32,
    neg_samples=4,
)
bpr_server = FedNCF(
    predictive_factor=4,
    interaction_dataset=dataset,
    client_algorithm=bpr_client,
    num_rounds=2,
    seed=SEED,
    client_selection_strategy=RandomClientSelection(dataset.num_clients, 4, "num"),
    aggregation_strategies={
        "item_embeddings": SimpleAvg(num_clients=dataset.num_clients),
        "shared_layers": FedAvg(num_clients=dataset.num_clients),
        "output_weights": FedAvg(num_clients=dataset.num_clients),
    },
    evaluator=None,
)
bpr_server.run()

last_losses = [
    loss
    for losses in bpr_server.current_round_statistics.training_losses.values()
    for loss in losses
]
print("last-round BPR losses:", last_losses)
assert last_losses and all(np.isfinite(last_losses))
assert set(bpr_server.aggregation_strategies) == {
    "item_embeddings",
    "shared_layers",
    "output_weights",
}

## Evaluate the pairwise run

`FullRankingEvaluator` scores, for each user, every catalogue item except items from the excluded splits, then averages HR and NDCG at 10. Test candidates drop that user's training and validation items. Scoring uses the client algorithm passed in, so this evaluator and the server share the pairwise client. Build a new evaluator for a new server.

The score uses the parameters training finished with. `current_round` is the last completed round. The result is a dict with keys `ndcg@10` and `hr@10`. Each value is the mean over users who have test interactions. On 20 items, HR@10 is a large fraction of the catalogue, so the number is a check that the run produced a score. The evaluator ranks the raw scores from `compute_scores`.

In [ ]:
from fedreclab.data.evaluator import FullRankingEvaluator

bpr_evaluator = FullRankingEvaluator(
    metrics=["ndcg", "hr"],
    topk=10,
    interaction_dataset=dataset,
    client_algorithm=bpr_client,
)
bpr_metrics = bpr_evaluator.evaluate(
    current_round=bpr_server.current_round,
    global_parameters=bpr_server.global_parameters,
    client_states=bpr_server.client_states,
    extra_payload=bpr_server.extra_payload,
    split="test",
)
print(bpr_metrics)
assert set(bpr_metrics) == {"ndcg@10", "hr@10"}
assert all(np.isfinite(list(bpr_metrics.values())))

## Layers that stay on the client

`shared_layers` and `output_weights` are dense state dicts. This variant keeps each client's trained copy in that client's state. `item_embeddings` stays on the server: a client upload of that group is the rows the client trained, while a client-held copy is the full table.

`PersonalisedFedNCF` lets the parent build all three groups, so every client can start from the same tensors. It then takes the chosen groups out of `global_parameters`, sets their `ParamIO` to `residency="client"`, `exchange="none"` and `apply_to=None`, and drops them from the aggregator map. The initial tensors are stored in `extra_payload["initial_local_groups"]`.

That dict stands in for a one-time download of the initial layers. A client that has not trained yet reads it during scoring. After a client trains, later rounds read the copy in its state.

The names are checked before the parent constructor runs. A rejected `local_groups` leaves the client, the selector and the aggregators unbound, so they can be passed to another server.

Pass an aggregator for every group that stays on the server. Here that is `item_embeddings`. An aggregator for a group that this class then moves is dropped during construction, so a dict copied from a plain FedNCF server works as well.

The pairwise client replaces `local_train` with its own loop. This variant wraps the library's pointwise `local_train`. Those are two overrides of the same method. A client that wants the pairwise loss and client-held layers uses this wrapper and puts the pairwise step inside it.

In [ ]:
from fedreclab.federated.utils import ParamIO
from fedreclab.methods.fedncf.fedncf import FedNCF, default_param_ios

class PersonalisedFedNCF(FedNCF):
    """FedNCF with chosen parameter groups kept on the client.

    ``item_embeddings`` stays server-side and aggregated. Every name in
    ``local_groups`` is removed from the server state after the parent
    initialises it, declared ``residency="client"`` and ``exchange="none"``,
    and dropped from the aggregator map. The parent initialisation of those
    groups is kept so each client can start from the same tensors.

    Names are checked before the parent constructor runs. A rejected
    ``local_groups`` leaves the client, the selector and the aggregators
    unbound, so they can be passed to another server.

    Simulation shortcuts:
        The shared initialisation is stored in
        ``extra_payload["initial_local_groups"]`` and read by any client
        that does not yet have its own copy. That stands in for a one-time
        download of the initial layers.

    Parameters
    ----------
    local_groups : sequence of str, optional
        FedNCF groups to keep on the client. Default is
        ``("shared_layers", "output_weights")``. ``item_embeddings`` is
        rejected because its upload is a sparse row set, and a client copy
        is a full table.
    *args, **kwargs
        Forwarded to ``FedNCF``. ``predictive_factor`` must match the client.

    Raises
    ------
    ValueError
        If a name is unknown, repeated, or is ``item_embeddings``.

    Examples
    --------
    Build with the same arguments as ``FedNCF``, and pass an aggregator
    for ``item_embeddings``. Afterwards ``global_parameters`` holds
    ``item_embeddings`` only, and ``extra_payload["initial_local_groups"]``
    holds the other two groups.
    """

    def __init__(self, *args, local_groups=("shared_layers", "output_weights"), **kwargs):
        self.local_groups = tuple(local_groups)
        # Reject unknown names before the parent binds the client.
        self._check_local_groups()
        super().__init__(*args, **kwargs)
        self._install_local_groups()

    def _check_local_groups(self) -> None:
        """Reject unknown, repeated, or row-layout group names.

        Returns
        -------
        None

        Raises
        ------
        ValueError
            If ``local_groups`` names an unknown group, repeats a name, or
            includes ``item_embeddings``.
        """
        known = set(default_param_ios())
        unknown = [name for name in self.local_groups if name not in known]
        if unknown:
            raise ValueError(
                f"unknown FedNCF group(s) {unknown}; known groups are {sorted(known)}"
            )
        if len(self.local_groups) != len(set(self.local_groups)):
            raise ValueError(f"local_groups has duplicates: {self.local_groups}")
        if "item_embeddings" in self.local_groups:
            raise ValueError(
                "item_embeddings is uploaded as sparse rows and cannot be kept "
                "client-resident by this server."
            )

    def _install_local_groups(self) -> None:
        """Move ``local_groups`` out of the server and publish their init.

        Returns
        -------
        None
        """
        initial = {}
        for name in self.local_groups:
            group = self.global_parameters.pop(name)
            initial[name] = {
                key: value.detach().cpu().clone() for key, value in group.items()
            }
            self.param_ios[name] = ParamIO(
                residency="client", exchange="none", apply_to=None
            )
            self.aggregation_strategies.pop(name, None)
        self.extra_payload["initial_local_groups"] = initial

## Item embeddings stay on the server

`local_groups=("item_embeddings",)` raises. The message says why. The client, selector and aggregator built for this call are discarded. The training cell builds new ones.

In [ ]:
try:
    PersonalisedFedNCF(
        predictive_factor=4,
        local_groups=("item_embeddings",),
        interaction_dataset=dataset,
        client_algorithm=FedNCFClientAlgorithm(predictive_factor=4),
        num_rounds=1,
        client_selection_strategy=RandomClientSelection(dataset.num_clients, 1, "num"),
        aggregation_strategies={
            "item_embeddings": SimpleAvg(num_clients=dataset.num_clients),
        },
    )
except ValueError as exc:
    print("rejected:", exc)
else:
    raise AssertionError("item_embeddings should be rejected")

## The client that keeps the layers

The parent `local_train` loads every group from `global_payload` and uploads every group. This subclass copies the client's layers into that payload, calls the parent, then moves those groups from the upload onto the client state. The server rejects a package whose group has `exchange="none"`, which is why those groups leave the upload after the parent has trained them.

`compute_scores` loads the same copy. The parent scores from `global_parameters` alone, and after this change that dict holds the item embeddings.

Adam moments for the MLP and the prediction layer follow the parent scope `client_resident`, which stores moments for the user tables. Those two layers start each local round from a fresh Adam state, as they do when the server holds them and the upload carries the weights.

In [ ]:
class PersonalisedFedNCFClientAlgorithm(FedNCFClientAlgorithm):
    """FedNCF client that trains groups the server no longer exchanges.

    The parent ``local_train`` loads a full skeleton and uploads every
    group. This subclass puts the client's copy of each local group into
    the payload before that call, then stores the trained copy on the
    client and removes it from the upload. ``compute_scores`` does the
    same load so ranking uses the client's layers. A client that has never
    trained uses ``extra_payload["initial_local_groups"]``.

    The group names are the keys of that initialisation, so they follow
    whatever ``PersonalisedFedNCF`` moved off the server. Adam moments for
    those layers are not persisted: the parent scope ``client_resident``
    keeps moments for the user tables only, which is also how those layers
    are trained when they live on the server.

    Examples
    --------
    Pair with ``PersonalisedFedNCF``. The constructor arguments are the
    ``FedNCFClientAlgorithm`` arguments.
    """

    def _initial_local_groups(self, extra_payload: dict | None) -> dict:
        """Return the shared initialisation published by the server.

        Parameters
        ----------
        extra_payload : dict or None
            Payload from the training request or from evaluation.

        Returns
        -------
        dict
            Group name to dict of tensors.

        Raises
        ------
        KeyError
            If the server did not publish ``initial_local_groups``.
        """
        init = None if extra_payload is None else extra_payload.get("initial_local_groups")
        if not init:
            raise KeyError(
                "extra_payload['initial_local_groups'] is missing. "
                "PersonalisedFedNCF publishes the shared initialisation there."
            )
        return init

    def _groups_for_client(
        self, client_state: ClientStateBase, extra_payload: dict | None
    ) -> dict:
        """Return this client's local groups, or the shared init if absent.

        Parameters
        ----------
        client_state : ClientStateBase
            State whose ``local_parameters`` may already store trained groups.
        extra_payload : dict or None
            Must contain ``initial_local_groups`` for groups the client has
            not stored yet.

        Returns
        -------
        dict
            Group name to parameter dict, one entry per local group.
        """
        init = self._initial_local_groups(extra_payload)
        stored = client_state.local_parameters
        return {name: stored[name] if name in stored else init[name] for name in init}

    def local_train(self, training_request: ClientTrainingRequest) -> ClientTrainingResponse:
        """Train with client-held groups and keep them out of the upload.

        Parameters
        ----------
        training_request : ClientTrainingRequest
            Standard FedNCF request. ``global_payload`` contains the groups
            the server still sends. ``extra_payload`` contains the shared
            initialisation of the groups it does not send.

        Returns
        -------
        ClientTrainingResponse
            Parent response with each local group moved from
            ``updates_packages`` onto ``updated_client_state``.

        Raises
        ------
        KeyError
            If the parent upload is missing a local group, or the shared
            initialisation was not published.
        """
        groups = self._groups_for_client(
            training_request.client_state, training_request.extra_payload
        )
        training_request.global_payload = {**training_request.global_payload, **groups}
        response = super().local_train(training_request)
        local = response.updated_client_state.local_parameters
        for name in groups:
            try:
                package = response.updates_packages.pop(name)
            except KeyError as exc:
                raise KeyError(
                    f"local group {name!r} was not in the training upload"
                ) from exc
            local[name] = package.param_value
        return response

    def compute_scores(
        self,
        client_id: int,
        candidate_items_per_user: dict,
        global_parameters: dict,
        client_state: ClientStateBase,
        extra_payload: dict,
    ) -> dict:
        """Score candidates with this client's local groups loaded.

        Parameters
        ----------
        client_id : int
            Client being scored.
        candidate_items_per_user : dict
            User id to candidate item ids. Forwarded to the parent.
        global_parameters : dict
            Server groups. Local groups are added from ``client_state`` or
            from the shared initialisation before the parent loads them.
        client_state : ClientStateBase
            User tables and, after the client has trained, the local groups.
        extra_payload : dict
            Must contain ``initial_local_groups`` for clients that have not
            trained.

        Returns
        -------
        dict
            User id to score array, as returned by the parent.
        """
        merged = dict(global_parameters)
        merged.update(self._groups_for_client(client_state, extra_payload))
        return super().compute_scores(
            client_id,
            candidate_items_per_user,
            merged,
            client_state,
            extra_payload,
        )

## Train with the layers on the client

This run needs a new client, a new selector and a new aggregator. The dataset is the one built above. The draw is again 4 of the 12 clients, for two rounds.

Before `run()`, the server parameters and the aggregator map contain `item_embeddings` only, and `shared_layers` is client-resident with `exchange="none"`. After `run()`, a selected client's state holds `shared_layers` and `output_weights`. A client that was never selected keeps the user embeddings created at the start, and scoring reads the MLP and the prediction layer from `extra_payload`. The prediction layer of a client that trained differs from that shared initialisation.

In [ ]:
personalised_client = PersonalisedFedNCFClientAlgorithm(
    predictive_factor=4,
    local_epochs=1,
    lr=5e-4,
    batch_size=32,
    neg_samples=4,
)
personalised_server = PersonalisedFedNCF(
    predictive_factor=4,
    interaction_dataset=dataset,
    client_algorithm=personalised_client,
    num_rounds=2,
    seed=SEED,
    client_selection_strategy=RandomClientSelection(dataset.num_clients, 4, "num"),
    aggregation_strategies={
        "item_embeddings": SimpleAvg(num_clients=dataset.num_clients),
    },
    evaluator=None,
)
print("server groups before training:", sorted(personalised_server.global_parameters))
print("aggregated:", sorted(personalised_server.aggregation_strategies))
assert set(personalised_server.global_parameters) == {"item_embeddings"}
assert set(personalised_server.aggregation_strategies) == {"item_embeddings"}
shared_io = personalised_server.param_ios["shared_layers"]
assert shared_io.residency == "client" and shared_io.exchange == "none"

personalised_server.run()

trained = [
    cid
    for cid, n in enumerate(personalised_server.participation_counts)
    if n > 0
]
idle = [
    cid
    for cid, n in enumerate(personalised_server.participation_counts)
    if n == 0
]
print("trained:", trained)
print("not selected:", idle)
assert trained and idle

state = personalised_server.client_states[trained[0]]
assert "shared_layers" in state.local_parameters
assert "output_weights" in state.local_parameters

idle_state = personalised_server.client_states[idle[0]]
assert "shared_layers" not in idle_state.local_parameters
assert "output_weights" not in idle_state.local_parameters

init_out = personalised_server.extra_payload["initial_local_groups"]["output_weights"]
trained_out = state.local_parameters["output_weights"]["output_logits.weight"]
assert trained_out.shape == init_out["output_logits.weight"].shape
delta = (trained_out.cpu() - init_out["output_logits.weight"].cpu()).abs().max().item()
print(f"prediction-layer max abs change: {delta:.6f}")
assert delta > 0

## Evaluate with the client's layers

The evaluator is given the personalised client, because `compute_scores` is what loads the local layers. Pass `extra_payload` so a client that has not trained can read the initial layers. The test score includes those clients.

On 20 items, HR@10 is a large fraction of the catalogue, so the number is a check that scoring ran for every user with a test interaction, including clients that were never selected.

In [ ]:
personalised_evaluator = FullRankingEvaluator(
    metrics=["ndcg", "hr"],
    topk=10,
    interaction_dataset=dataset,
    client_algorithm=personalised_client,
)
personalised_metrics = personalised_evaluator.evaluate(
    current_round=personalised_server.current_round,
    global_parameters=personalised_server.global_parameters,
    client_states=personalised_server.client_states,
    extra_payload=personalised_server.extra_payload,
    split="test",
)
print(personalised_metrics)
assert set(personalised_metrics) == {"ndcg@10", "hr@10"}
assert all(np.isfinite(list(personalised_metrics.values())))